# Benchmark Results — LLM Comparison

Visualizes benchmark suite results from `benchmark.py` runs, comparing models on cost, speed, token usage, retries, and lab quality metrics.

In [ ]:
import json
import glob
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

BENCHMARK_DIR = Path("../../Artifacts/Data/Benchmark")

# Load the most recent suite_results file
suite_files = sorted(BENCHMARK_DIR.glob("suite_results_*.json"))
if not suite_files:
    raise FileNotFoundError(f"No suite_results_*.json found in {BENCHMARK_DIR}")

latest = suite_files[-1]
print(f"Loading: {latest.name}")
raw = json.loads(latest.read_text(encoding="utf-8"))

In [ ]:
# Flatten nested JSON into a DataFrame
rows = []
for r in raw:
    tracking = r.get("tracking", {})
    tokens = tracking.get("tokens", {})
    retries = tracking.get("retries", {})
    errors = tracking.get("errors", {})
    cost = tracking.get("cost", {})
    lab = r.get("lab_metrics", {})

    rows.append({
        "Model": r.get("display_name", r.get("model", "?")),
        "Provider": r.get("provider", "?"),
        "Success": r.get("success", False),
        "Wall Time (s)": r.get("wall_time_seconds", 0),
        "Prompt Tokens": tokens.get("prompt", 0),
        "Completion Tokens": tokens.get("completion", 0),
        "Total Tokens": tokens.get("total", 0),
        "Cost ($)": cost.get("total_usd", 0),
        "Retries": retries.get("total", 0),
        "Parse Errors": errors.get("parse_errors", 0),
        "Objects": lab.get("num_objects", 0),
        "Clips": lab.get("num_clips", 0),
        "Components": lab.get("num_components", 0),
        "Object Changes": lab.get("num_object_changes", 0),
        "Unique Prefabs": lab.get("num_unique_prefabs", 0),
        "Text Labels": lab.get("num_text_labels", 0),
    })

df = pd.DataFrame(rows).set_index("Model")
df

## Summary Table

In [ ]:
summary_cols = ["Provider", "Success", "Wall Time (s)", "Total Tokens", "Cost ($)", "Retries", "Parse Errors"]
summary = df[summary_cols].copy()
summary["Cost ($)"] = summary["Cost ($)"].map("${:.6f}".format)
summary["Wall Time (s)"] = summary["Wall Time (s)"].map("{:.1f}".format)
summary.style.map(
    lambda v: "background-color: #d4edda" if v is True else ("background-color: #f8d7da" if v is False else ""),
    subset=["Success"]
)

## Cost & Speed Comparison

In [ ]:
PROVIDER_COLORS = {"openai": "#10a37f", "anthropic": "#d4a574", "google": "#4285f4"}
colors = [PROVIDER_COLORS.get(p, "#999") for p in df["Provider"]]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Cost
ax = axes[0]
bars = ax.barh(df.index, df["Cost ($)"], color=colors)
ax.set_xlabel("Cost (USD)")
ax.set_title("Cost per Run")
ax.xaxis.set_major_formatter(mtick.FormatStrFormatter("$%.4f"))
for bar, v in zip(bars, df["Cost ($)"]):
    ax.text(v + max(df["Cost ($)"]) * 0.02, bar.get_y() + bar.get_height() / 2,
            f"${v:.4f}", va="center", fontsize=9)

# Wall time
ax = axes[1]
bars = ax.barh(df.index, df["Wall Time (s)"], color=colors)
ax.set_xlabel("Seconds")
ax.set_title("Wall Time")
for bar, v in zip(bars, df["Wall Time (s)"]):
    ax.text(v + max(df["Wall Time (s)"]) * 0.02, bar.get_y() + bar.get_height() / 2,
            f"{v:.1f}s", va="center", fontsize=9)

# Total tokens
ax = axes[2]
bars = ax.barh(df.index, df["Total Tokens"], color=colors)
ax.set_xlabel("Tokens")
ax.set_title("Total Tokens")
for bar, v in zip(bars, df["Total Tokens"]):
    ax.text(v + max(df["Total Tokens"]) * 0.02, bar.get_y() + bar.get_height() / 2,
            f"{v:,}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## Token Breakdown (Prompt vs Completion)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

models = df.index.tolist()
x = range(len(models))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], df["Prompt Tokens"], width, label="Prompt", color="#5b9bd5")
bars2 = ax.bar([i + width/2 for i in x], df["Completion Tokens"], width, label="Completion", color="#ed7d31")

ax.set_ylabel("Tokens")
ax.set_title("Token Usage by Model")
ax.set_xticks(list(x))
ax.set_xticklabels(models, rotation=15, ha="right")
ax.legend()
ax.bar_label(bars1, fmt="{:,.0f}", fontsize=8, padding=2)
ax.bar_label(bars2, fmt="{:,.0f}", fontsize=8, padding=2)

plt.tight_layout()
plt.show()

## Retries & Errors

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

x = range(len(models))
width = 0.35

ax.bar([i - width/2 for i in x], df["Retries"], width, label="Total Retries", color="#ffc107")
ax.bar([i + width/2 for i in x], df["Parse Errors"], width, label="Parse Errors", color="#dc3545")

ax.set_ylabel("Count")
ax.set_title("Retries & Parse Errors")
ax.set_xticks(list(x))
ax.set_xticklabels(models, rotation=15, ha="right")
ax.legend()
ax.yaxis.set_major_locator(mtick.MaxNLocator(integer=True))

plt.tight_layout()
plt.show()

## Lab Quality Metrics (Successful Runs Only)

In [ ]:
ok = df[df["Success"] == True].copy()

if ok.empty:
    print("No successful runs to compare.")
else:
    quality_cols = ["Objects", "Clips", "Components", "Object Changes", "Unique Prefabs", "Text Labels"]
    ok_quality = ok[quality_cols]

    fig, ax = plt.subplots(figsize=(12, 5))
    ok_quality.plot(kind="bar", ax=ax, width=0.75)
    ax.set_title("Lab Structural Metrics — Successful Runs")
    ax.set_ylabel("Count")
    ax.set_xticklabels(ok.index, rotation=15, ha="right")
    ax.legend(loc="upper left", fontsize=9)
    ax.yaxis.set_major_locator(mtick.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()

    # Quality summary table
    ok_quality

## Efficiency: Cost per Lab Element

In [ ]:
if not ok.empty:
    efficiency = pd.DataFrame(index=ok.index)
    total_elements = ok["Objects"] + ok["Clips"] + ok["Components"]
    efficiency["Total Elements"] = total_elements
    efficiency["Cost ($)"] = ok["Cost ($)"]
    efficiency["$ / Element"] = (ok["Cost ($)"] / total_elements.replace(0, float("nan")))
    efficiency["Tokens / Element"] = (ok["Total Tokens"] / total_elements.replace(0, float("nan"))).round(0).astype(int)
    efficiency["Seconds / Element"] = (ok["Wall Time (s)"] / total_elements.replace(0, float("nan"))).round(2)
    efficiency